# 06 · A SQL Data Agent — with a Gradio UI

**Where we are in the stack:** still the **Agent = control plane** from notebook
2. New resource: a **SQLite database**. The agent answers questions about the
data by inspecting the schema and running **read-only** `SELECT` queries.

Same loop:

```
prompt the model  ->  did it ask for a tool?
      ^                       |  yes                 |  no
      |                  call the tool          FINAL ANSWER -> stop
      |                       |
      +----- feed result back +
```

Input: a **question** about the data. The agent must `list_tables`,
`describe_table`, then `run_select` - it cannot make up rows.

> Only `SELECT` is allowed. Writes (INSERT/UPDATE/DELETE/DROP) are rejected -
> the read-only default from notebook 2, enforced for SQL.

## 0. Provider config (loads `.env`)

Needs a **tool-capable** model. Settings come from `.env`.

In [ ]:
# --- Provider config: loads .env, works with any OpenAI-compatible server ---
import os
from openai import OpenAI

# Load .env if python-dotenv is available; otherwise parse it by hand.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    if os.path.exists(".env"):
        for line in open(".env"):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. A sample database (so the demo is self-contained)

A tiny network-inventory DB: `devices` and `interfaces`. Point the agent at any
SQLite file instead by changing `DB_PATH`.

In [ ]:
import os, sqlite3

DB_PATH = os.path.abspath("inventory.db")
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

con = sqlite3.connect(DB_PATH)
con.executescript("""
CREATE TABLE devices (
    id INTEGER PRIMARY KEY, hostname TEXT, role TEXT, site TEXT, uptime_days INTEGER);
CREATE TABLE interfaces (
    id INTEGER PRIMARY KEY, device_id INTEGER, name TEXT,
    oper_status TEXT, speed_gbps INTEGER, errors INTEGER);
INSERT INTO devices (hostname, role, site, uptime_days) VALUES
    ('leaf-01','leaf','blr', 120), ('leaf-02','leaf','blr', 12),
    ('spine-01','spine','blr', 300), ('spine-02','spine','del', 5);
INSERT INTO interfaces (device_id, name, oper_status, speed_gbps, errors) VALUES
    (1,'et-0/0/1','up',10,0), (1,'et-0/0/2','down',10,42),
    (2,'et-0/0/1','up',25,3), (3,'et-0/0/1','up',100,0),
    (3,'et-0/0/2','up',100,7), (4,'et-0/0/1','down',25,15);
""")
con.commit(); con.close()
print("created", DB_PATH)

## 2. The tools = read-only database access

Three functions. `run_select` guards the read-only rule: it parses the statement
and refuses anything that isn't a single `SELECT`.

In [ ]:
import sqlite3

def _connect():
    return sqlite3.connect(DB_PATH)

def list_tables():
    """List user tables in the database."""
    con = _connect()
    rows = con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    con.close()
    return {"tables": [r[0] for r in rows]}

def describe_table(table):
    """Return column names and types for a table (like PRAGMA table_info)."""
    con = _connect()
    try:
        rows = con.execute(f"PRAGMA table_info({table})").fetchall()
    except sqlite3.Error as e:
        con.close(); return {"error": str(e)}
    con.close()
    return {"table": table,
            "columns": [{"name": r[1], "type": r[2]} for r in rows]}

def run_select(query, max_rows=50):
    """Run a single read-only SELECT and return rows. Writes are rejected."""
    q = query.strip().rstrip(";")
    low = q.lower()
    if not low.startswith("select"):
        return {"error": "only SELECT statements are allowed"}
    forbidden = (" insert ", " update ", " delete ", " drop ", " alter ",
                 " create ", " replace ", " attach ", " pragma ")
    if any(tok in f" {low} " for tok in forbidden) or ";" in q:
        return {"error": "statement contains a non-read-only or chained command"}
    con = _connect()
    try:
        cur = con.execute(q)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchmany(max_rows)
    except sqlite3.Error as e:
        con.close(); return {"error": str(e)}
    con.close()
    return {"columns": cols, "rows": [list(r) for r in rows], "row_count": len(rows)}

# Prove the tools work without the model
print(list_tables())
print(describe_table("interfaces"))
print(run_select("SELECT hostname, role FROM devices LIMIT 2"))
print(run_select("DELETE FROM devices"))   # blocked

## 3. Describe the tools to the model (JSON schema)

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "list_tables",
        "description": "List the tables in the database.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "describe_table",
        "description": "Return the columns and types of a table.",
        "parameters": {"type": "object",
            "properties": {"table": {"type": "string"}},
            "required": ["table"]}}},
    {"type": "function", "function": {
        "name": "run_select",
        "description": "Run a single read-only SELECT query and return rows.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string", "description": "a SELECT statement"}},
            "required": ["query"]}}},
]

TOOL_REGISTRY = {"list_tables": list_tables,
                 "describe_table": describe_table, "run_select": run_select}

## 4. The agent loop (the same FSM as notebook 2)

In [ ]:
import json

SYSTEM = """You are a data analyst with read-only access to a SQL database.
Answer ONLY from query results, using your tools.
Strategy: list_tables -> describe_table for the relevant tables -> write a SELECT
(use JOINs/aggregates as needed) -> run_select -> interpret the rows.
Show the query you ran and the numbers behind your answer. Never invent data."""

def ask_data(question, max_iterations=12, temperature=0):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    print("DB:", DB_PATH); print("Q:", question); print("=" * 72)
    for step in range(1, max_iterations + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=temperature)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            print(f"\n[step {step}] FINAL ANSWER\n{'-'*72}\n{msg.content}")
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            name = tc.function.name; args = json.loads(tc.function.arguments)
            print(f"[step {step}] TOOL  -> {name}({args})")
            try: result = TOOL_REGISTRY[name](**args)
            except Exception as e: result = {"error": str(e)}
            preview = str(result)
            print(f"[step {step}] RESULT <- {preview[:200]}{'...' if len(preview) > 200 else ''}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result)[:4000]})
    return "Stopped: hit max_iterations (TTL expired)."

## 5. Run it - a question that needs a JOIN

In [ ]:
_ = ask_data("Which devices have any interface that is down, and how many errors are on those interfaces?")

## 6. Run it - an aggregate question

In [ ]:
_ = ask_data("What is the total interface error count per site? Which site is worst?")

## Recap

Fourth agent, **same loop** - now over a database, with `SELECT`-only access as
the safety gate. Across notebooks 02, 04, 05 and 06 the control plane never
changed; only the **capability table** did:

| Agent | Resource | Tools |
|---|---|---|
| 02 from scratch | network (mocked) | subnet calc, interface status |
| 04 codebase explainer | filesystem | ls, cat, grep, find |
| 05 log triage | log files | ls, tail, grep, count |
| 06 SQL data | database | list_tables, describe, select |

That is the whole lesson: **an agent is a loop plus tools.** Pick the resource,
write a few read-only tools, keep the loop. Frameworks (notebook 3) then hand you
retries, streaming, memory/checkpointing and confirmation gates for free.

## 7. Interactive UI (Gradio)

Ask a question in plain English; the agent inspects the schema and runs a
read-only `SELECT`. The **Agent trace** box shows the SQL it wrote.

In [ ]:
# Install Gradio for the interactive UI (run once).
%pip install -q gradio

In [ ]:
import io, contextlib

def _run_capture(fn, *args, **kwargs):
    """Run an agent function, capturing its printed step-by-step trace."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        try:
            result = fn(*args, **kwargs)
        except Exception as e:
            result = f"Error: {e}"
    return buf.getvalue(), (result or "")

In [ ]:
import gradio as gr

def sql_ui(db_path, question):
    global DB_PATH
    DB_PATH = os.path.abspath(db_path or "inventory.db")
    trace, answer = _run_capture(ask_data, question)
    return trace, answer

with gr.Blocks(title="06 · SQL Data Agent") as demo:
    gr.Markdown("# 🗄️ SQL Data Agent\n"
                "Read-only database tools (list_tables / describe_table / run_select). "
                "`run_select` rejects anything that isn't a single SELECT.")
    db_path = gr.Textbox(label="SQLite database path", value="inventory.db")
    question = gr.Textbox(
        label="Question",
        value="Which devices have any interface that is down, and how many errors are on those interfaces?",
    )
    run = gr.Button("Ask", variant="primary")
    gr.Examples(
        ["Which devices have any interface that is down, and how many errors are on those interfaces?",
         "What is the total interface error count per site? Which site is worst?"],
        inputs=question,
    )
    answer = gr.Markdown(label="Answer")
    trace = gr.Textbox(label="Agent trace (SQL queries)", lines=16)
    run.click(sql_ui, inputs=[db_path, question], outputs=[trace, answer])

demo.launch()